In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
"""
Note: The master method sets the master URL for the Spark application.
"local[*]" means that Spark will run locally with as many worker threads as logical cores on your machine. 
This is useful for development and testing purposes.
You can also set it to other cluster managers like yarn, mesos, or provide a Spark standalone cluster URL.
"""
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
#Uncomment when required (note the files won't be committed due to the .gitignore

#!curl -L -O https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
#!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

In [4]:
#Uncomment when required
#!gzip -dc fhvhv_tripdata_2021-01.csv.gz

In [3]:
# word count
!wc -l fhvhv_tripdata_2021-01.csv

11908469 fhvhv_tripdata_2021-01.csv


In [4]:

df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [5]:
"""
Note:
Spark is reading the data as strings instead of timestamps or numbers. 
Unlike Pandas, Spark does not infer data types automatically, so everything is treated as a string by default.
The schema shows all fields are classified as string type.
"""
df.schema


StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [6]:
#top 20 records
df.show(5)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   null|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   null|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   null|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   null|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   null|
+-----------------+--------------------+-------------------+-------------------+

In [7]:
#top 5 records
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

In [8]:
# Create a new .csv file in the location and name it as "head.csv". Its 1000 rows
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [9]:
!wc -l head.csv

1001 head.csv


In [10]:
# Powerful data structures for data analysis, time series, and statistics 
# Reading this with pandas instead of spark for size of data
import pandas as pd

In [11]:
df_pandas = pd.read_csv('head.csv')

In [12]:
# Checking data types in pandas
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

Integer - 4 bytes
Long - 8 bytes

In [13]:
from pyspark.sql import types

In [ ]:
# Using spark to read the schema after creating a data frame from the pandas df we created (1001 rows of head.csv)
# Note: You might have an error indicating incompatibility between pandas and pyspark version of the dataframe conversion
# Solutions indicate to reinstall Spark in version 3.4.2 or newer. However we installed an older version due to write.parquet error encountered earlier.
# IGNORE THE ERROR as not relevant

spark.createDataFrame(df_pandas).schema

In [ ]:
# Solution to the error above even though it is a nice trick with pandas
# You can do the same with pyspark itself, with option("InferSchema", True).
# Some agree it is a more 'pyspark' way of doing that 


In [16]:
# You can even skip .option() method and pass everything with parameters shown below

df_test = spark.read.csv('head.csv', header=True, inferSchema=True)

In [20]:
# You now have your data frame created using spark
df_test.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [21]:
df_test.show(5)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   null|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   null|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   null|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   null|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   null|
+-----------------+--------------------+-------------------+-------------------+

In [39]:
# You now have your data frame created using spark
# Note that the "infer_schema" option used earlier transformed the df.test into the right data types
# Hence, no need to define schema as done below. But for the purpose of following the tutorial, we will do it for our main dataframe(df)
df_test.printSchema()


root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [26]:
# Defining the schema.
# We require the "types" module
"""
Defining the schema:
To properly define a schema for our DataFrame, I will format the inferred schema. 
Spark schemas use StructType, which is a Scala construct, so I need to convert it into Python code.
"""

schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [27]:
# After defining the schema, I need to specify it when reading the CSV file. 
# Adding the schema parameter ensures that Spark correctly interprets the data types.

df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [38]:
"""
Running df.head(10) on the loaded data confirms that timestamps are parsed correctly, 
location IDs are treated as numbers without quotes, 
and SR_Flag remains a nullable string.

Compare with the previous df.head to see results
"""
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02883', pickup_datetime=datetime.datetime(2021, 1, 2, 20, 27, 2), dropoff_datetime=datetime.datetime(2021, 1, 2, 20, 35, 57), PULocationID=249, DOLocationID=107, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02835', pickup_datetime=datetime.datetime(2021, 1, 2, 16, 38, 39), dropoff_datetime=datetime.datetime(2021, 1, 2, 16, 49, 56), PULocationID=225, DOLocationID=62, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02765', pickup_datetime=datetime.datetime(2021, 1, 3, 15, 53, 25), dropoff_datetime=datetime.datetime(2021, 1, 3, 15, 57, 36), PULocationID=123, DOLocationID=123, SR_Flag=None),
 Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 2, 0, 46, 7), dropoff_datetime=datetime.datetime(2021, 1, 2, 0, 59), PULocationID=143, DOLocationID=24, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02887', pickup_dat

In [29]:
# Note that we cannot see the difference if we use df.show() 
# df.head() helps to see the data type changes instead
df.show(5)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   null|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   null|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   null|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   null|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   null|
+-----------------+--------------------+-------------------+-------------------+

In [30]:
# To be precise run this to show the new changes:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [31]:
# PARTITIONS
"""
Executing df.repartition(24) does not immediately change the DataFrame because repartitioning is lazy. 
The change is applied only when we perform an action, such as saving the DataFrame.
"""

'\nExecuting df.repartition(24) does not immediately change the DataFrame because repartitioning is lazy. \nThe change is applied only when we perform an action, such as saving the DataFrame.\n'

In [32]:
df = df.repartition(24)

In [33]:
# Bring it back if required. This is the action that saves the data frame >> df.write.parquet('file_path_to_save_parquet')

# df.write.parquet('fhvhv/2021/01/')

In [34]:
# SPARK DATAFRAMES

In [35]:
# Note: We write to parquet for efficiency in its data storage
df = spark.read.parquet('fhvhv/2021/01/')

In [36]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



SELECT * FROM df WHERE hvfhs_license_num =  HV0003

In [44]:
# Selecting just few columns
df.select("pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID").show(5)


+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-02 20:27:02|2021-01-02 20:35:57|         249|         107|
|2021-01-02 16:38:39|2021-01-02 16:49:56|         225|          62|
|2021-01-03 15:53:25|2021-01-03 15:57:36|         123|         123|
|2021-01-02 00:46:07|2021-01-02 00:59:00|         143|          24|
|2021-01-02 18:27:36|2021-01-02 19:07:42|         132|         265|
+-------------------+-------------------+------------+------------+
only showing top 5 rows



In [45]:
# This still shows us the entire table
df.show(5)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02883|2021-01-02 20:27:02|2021-01-02 20:35:57|         249|         107|   null|
|           HV0003|              B02835|2021-01-02 16:38:39|2021-01-02 16:49:56|         225|          62|   null|
|           HV0003|              B02765|2021-01-03 15:53:25|2021-01-03 15:57:36|         123|         123|   null|
|           HV0005|              B02510|2021-01-02 00:46:07|2021-01-02 00:59:00|         143|          24|   null|
|           HV0003|              B02887|2021-01-02 18:27:36|2021-01-02 19:07:42|         132|         265|   null|
+-----------------+--------------------+-------------------+-------------------+

In [50]:
# FILTERING DATA
# Filter statement to get only the records where a specific license number matches a given value
# To add another filter use>> .filter(df.´column_name')
# Note we add show() for spark to actually do something >> Action. 
df.select('hvfhs_license_num', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003') \
  .filter(df.PULocationID == '249') \
  .show(5)


+-----------------+-------------------+-------------------+------------+------------+
|hvfhs_license_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-----------------+-------------------+-------------------+------------+------------+
|           HV0003|2021-01-02 20:27:02|2021-01-02 20:35:57|         249|         107|
|           HV0003|2021-01-04 04:56:28|2021-01-04 05:03:41|         249|         230|
|           HV0003|2021-01-05 11:14:25|2021-01-05 11:43:52|         249|         236|
|           HV0003|2021-01-03 09:32:10|2021-01-03 09:39:12|         249|          68|
|           HV0003|2021-01-03 12:43:10|2021-01-03 12:50:40|         249|          79|
+-----------------+-------------------+-------------------+------------+------------+
only showing top 5 rows



In [30]:
# SPARK ACTIONS examples
"""
show(): Displays the DataFrame.
take(5): Retrieves the first five records. Similar to head()
write.csv() or write.parquet() – Triggers execution to write results to storage.
"""

df.take(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02883', pickup_datetime=datetime.datetime(2021, 1, 2, 20, 27, 2), dropoff_datetime=datetime.datetime(2021, 1, 2, 20, 35, 57), PULocationID=249, DOLocationID=107, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02835', pickup_datetime=datetime.datetime(2021, 1, 2, 16, 38, 39), dropoff_datetime=datetime.datetime(2021, 1, 2, 16, 49, 56), PULocationID=225, DOLocationID=62, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02765', pickup_datetime=datetime.datetime(2021, 1, 3, 15, 53, 25), dropoff_datetime=datetime.datetime(2021, 1, 3, 15, 57, 36), PULocationID=123, DOLocationID=123, SR_Flag=None),
 Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 2, 0, 46, 7), dropoff_datetime=datetime.datetime(2021, 1, 2, 0, 59), PULocationID=143, DOLocationID=24, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02887', pickup_dat

In [31]:
# FUNCITONS example to create a new column using the "df.withColumn"
"""
So why use Spark instead? Spark is more flexible and provides additional functionality, such as User-Defined Functions (UDFs).
In Spark, we have pyspark.sql.functions, a collection of functions that Spark provides (Built-in functions). 
To use them you need the following:
"""

'\nSo why use Spark instead? Spark is more flexible and provides additional functionality, such as User-Defined Functions (UDFs).\nIn Spark, we have pyspark.sql.functions, a collection of functions that Spark provides (Built-in functions). \nTo use them you need the following:\n'

In [51]:
from pyspark.sql import functions as F

In [52]:
# Note here that df.show at the end brings every column out. to select specific ones, use df.select 

df \
    .withColumn("pickup_date", F.to_date(df.pickup_datetime)) \
    .withColumn("dropoff_date", F.to_date(df.dropoff_datetime)) \
    .show(5)    

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|           HV0003|              B02883|2021-01-02 20:27:02|2021-01-02 20:35:57|         249|         107|   null| 2021-01-02|  2021-01-02|
|           HV0003|              B02835|2021-01-02 16:38:39|2021-01-02 16:49:56|         225|          62|   null| 2021-01-02|  2021-01-02|
|           HV0003|              B02765|2021-01-03 15:53:25|2021-01-03 15:57:36|         123|         123|   null| 2021-01-03|  2021-01-03|
|           HV0005|              B02510|2021-01-02 00:46:07|2021-01-02 00:59:00|         143|          24|   null| 2021-01-02|  2021-01-02|
|           HV0003| 

In [78]:
# Here, notice how we mix the new columns created with already existing columns.
# Thanks to .withColumn; .select().
# Note we cannot use the .select without the new column being defined. Hence, it goes at the end
# Similar to your SQL order of SELECT, use the filter at end
df \
    .withColumn("pickup_date", F.to_date(df.pickup_datetime)) \
    .withColumn("dropoff_date", F.to_date(df.dropoff_datetime)) \
    .select("hvfhs_license_num", "dispatching_base_num", "pickup_date", "dropoff_date", "PULocationID", "DOLocationID", "SR_Flag") \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .filter(df.PULocationID == '249') \
    .show(5)   


+-----------------+--------------------+-----------+------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|pickup_date|dropoff_date|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-----------+------------+------------+------------+-------+
|           HV0003|              B02883| 2021-01-02|  2021-01-02|         249|         107|   null|
|           HV0003|              B02872| 2021-01-04|  2021-01-04|         249|         230|   null|
|           HV0003|              B02870| 2021-01-05|  2021-01-05|         249|         236|   null|
|           HV0003|              B02875| 2021-01-03|  2021-01-03|         249|          68|   null|
|           HV0003|              B02879| 2021-01-03|  2021-01-03|         249|          79|   null|
+-----------------+--------------------+-----------+------------+------------+------------+-------+
only showing top 5 rows



In [79]:
# Original data frame
df.show(5)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02883|2021-01-02 20:27:02|2021-01-02 20:35:57|         249|         107|   null|
|           HV0003|              B02835|2021-01-02 16:38:39|2021-01-02 16:49:56|         225|          62|   null|
|           HV0003|              B02765|2021-01-03 15:53:25|2021-01-03 15:57:36|         123|         123|   null|
|           HV0005|              B02510|2021-01-02 00:46:07|2021-01-02 00:59:00|         143|          24|   null|
|           HV0003|              B02887|2021-01-02 18:27:36|2021-01-02 19:07:42|         132|         265|   null|
+-----------------+--------------------+-------------------+-------------------+

In [35]:
# User-Defined Functions
"""
Let's say we have a function that performs complex logic, something not easy to express with SQL. 
I.e, it processes a column called dispatching_base_number, 
Extracts the numeric part of the string by removing the first character (base_num[1:]) and converts it to an integer (num).
And it does string formatting based on the logic divisible provided
"""

"""
Expressing this in SQL would be cumbersome, especially as the logic grows more complex with multiple conditions. 
The advantage of implementing this logic in Python is that it can live in a separate module and it can be unit-tested.
"""

'\nExpressing this in SQL would be cumbersome, especially as the logic grows more complex with multiple conditions. \nThe advantage of implementing this logic in Python is that it can live in a separate module and it can be unit-tested.\n'

In [80]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [86]:
#Testing
crazy_stuff('B02884')

's/b44'

In [96]:
# Testing 2
crazy_stuff('49')

'a/009'

In [98]:
# Now, to turn this Python function into a User-Defined Function (UDF) in PySpark:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [105]:
# checking the python version because of the error we get below >> PicklingError: Could not serialize object: IndexError: tuple index out of range

!python --version

Python 3.12.0


In [107]:
!pyspark --version

# Our version is 3.3.2

Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.3.2
      /_/
                        
Using Scala version 2.12.15, Java HotSpot(TM) 64-Bit Server VM, 11.0.25
Branch HEAD
Compiled by user liangchi on 2023-02-10T19:18:32Z
Revision 5103e00c4ce5fcc4264ca9c4df12295d42557af6
Url https://github.com/apache/spark
Type --help for more information.


In [118]:
# Finding out the pandas version
!python -c "import pandas as pd; print(pd.__version__)"

2.2.3


In [117]:
# Do the below in the terminal of VS code due to interactive mode
# Looks like this command gives an error anyways

#!pip uninstall pandas

In [ ]:
# Now we can use this udf:
# If you get the error its due to the python version you are using. Note this is just an example, just skip


df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show(5)

In [119]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,
